## Import the dependencies

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Load and inspect the dataset

In [ ]:
fraud_df = pd.read_csv('../data/raw/Fraud_Data.csv')
ip_country_df = pd.read_csv ('../data/raw/IpAddress_to_Country.csv')

In [ ]:
print(fraud_df.head())
## fraud_df.ip_address.dtypes
## fraud_df.signup_time.dtypes
##print(ip_country_df.head())

## print(fraud_df.info())
## print(fraud_df.shape)

# 2. Data Cleaning

### 2.1. Missing value checks

In [ ]:
print(fraud_df.isnull().sum())
print(fraud_df.isin(['', ' ', 'NaN', 'null']).sum())


### 2.2. Duplicates

In [ ]:
# Duplicates
print("Duplicate rows:", fraud_df.duplicated().sum())
print("Duplicate user_ids:", fraud_df['user_id'].duplicated().sum())

### 2.3. Data Types

In [ ]:
print(fraud_df.dtypes)

In [22]:
# Fix dtypes
fraud_df['signup_time'] = pd.to_datetime(fraud_df['signup_time'])
fraud_df['purchase_time'] = pd.to_datetime(fraud_df['purchase_time'])
fraud_df['ip_address'] = fraud_df['ip_address'].astype('int64')

ip_country_df['lower_bound_ip_address'] = (
    ip_country_df['lower_bound_ip_address'].astype('int64')
)

ip_country_df['upper_bound_ip_address'] = (
    ip_country_df['upper_bound_ip_address'].astype('int64')
)
print(fraud_df.dtypes)
print(ip_country_df.dtypes)

user_id                    int64
signup_time       datetime64[ns]
purchase_time     datetime64[ns]
purchase_value             int64
device_id                 object
source                    object
browser                   object
sex                       object
age                        int64
ip_address                 int64
class                      int64
dtype: object
lower_bound_ip_address     int64
upper_bound_ip_address     int64
country                   object
dtype: object


# 3. Exploratory Data Analysis (EDA)

### 3.1 Univariate Analysis
 - The key variable that I am going to consider are purchase_value, age, and source (browser). Why?
     - Skew in purchase_value and Amount — fraud transactions often sit at the unusual extremes (very small "test" charges or unusually large ones), so if the distribution has a long tail, that tail is worth a second look later in bivariate analysis.
     - Spread and range in age — are there implausible values (e.g. age 0 or age 120)? That's a data quality flag, not just a shape observation.
     - Category dominance in source and browser — if 95% of all traffic comes through one browser, then "browser" alone won't discriminate fraud well; if it's more balanced, it might carry real signal.

In [ ]:
print(fraud_df[['purchase_value' , 'age']].describe())

In [ ]:
print(fraud_df['source'].value_counts())

In [ ]:
print(fraud_df['sex'].value_counts())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Numeric summary
print(fraud_df[['purchase_value', 'age']].describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(fraud_df['purchase_value'], bins=50, ax=axes[0])
axes[0].set_title('Purchase Value Distribution')
sns.histplot(fraud_df['age'], bins=30, ax=axes[1])
axes[1].set_title('Age Distribution')
plt.tight_layout()
plt.show()

# Categorical counts
print(fraud_df['source'].value_counts())
print(fraud_df['browser'].value_counts())
print(fraud_df['sex'].value_counts())

## 3.2 Bivariate Analysis

In [ ]:
print(fraud_df['class'].value_counts())
print(fraud_df['class'].value_counts(normalize=True) * 100)

In [ ]:
# Fraud rate by numerical columns - Purchase_value & age by class
print(fraud_df.groupby('class')[['purchase_value', 'age']].mean())
print(fraud_df.groupby('class')[['purchase_value', 'age']].median())

In [ ]:
# fraud RATE by category - source, browser, sex but here mean() usage is different from the above. U can c the syntax
print(fraud_df.groupby('source')['class'].mean())

print(fraud_df.groupby('browser')['class'].mean())

print(fraud_df.groupby('sex')['class'].mean())


### Findings from Bivariate analysis
I have classified the analysis in 2 categories. Numerical and categorical. 
1. Numerical Features - 2 columns namely purcahse value and age were used to tell there is something between class and them. 36.93 vs 36.99 for purchase_value, 33.12 vs 33.32 for age - essentially identical between fraud and legitimate. This is a real, useful negative finding, not a failure: it tells you fraudsters in this dataset aren't distinguishing themselves by spending unusually or by age
2. Categorical Features - 2 columns namely source, browser, and sex were taken in to consideration.
- Source - Direct traffic has the highest fraud rate (10.5%) vs Ads (9.2%) and SEO (8.9%) - a real, if modest, difference. Worth flagging: fraudsters may disproportionately land via direct URL rather than coming through a tracked marketing channel.
- Browser: Chrome and FireFox sit slightly higher (~9.5–9.9%) than IE (8.7%) - small differences, nothing dramatic.
- Sex: 9.55% (M) vs 9.10% (F) - negligible difference, not something to build a narrative around.

Overall verdict for Fraud_Data's raw columns: weak, subtle signals at best - nothing screams "this is the fraud tell." That's expected and fine; it's exactly why the project has me engineer new features rather than relying on what's handed to me.

## 4. Geolocation Merge

In [ ]:
print(ip_country_df.columns)

In [ ]:
# print(fraud_df["ip_address"].dtypes)
print(ip_country_df["lower_bound_ip_address"].dtypes)

In [23]:
# Sort the country table by lower bound (required for merge_asof)
ip_country_df = ip_country_df.sort_values('lower_bound_ip_address').reset_index(drop=True)

# Sort fraud_df by ip_address (also required for merge_asof)
fraud_df_sorted = fraud_df.sort_values('ip_address').reset_index(drop=True)

# Range-based merge: find nearest lower_bound <= ip_address
merged = pd.merge_asof(
    fraud_df_sorted,
    ip_country_df,
    left_on='ip_address',
    right_on='lower_bound_ip_address',
    direction='backward'
)

# Verify the match actually falls within the upper bound too
merged['valid_match'] = merged['ip_address'] <= merged['upper_bound_ip_address']
print(merged['valid_match'].value_counts())

# Where invalid, country should be treated as unknown
merged.loc[~merged['valid_match'], 'country'] = 'Unknown'

fraud_df = merged.drop(columns=['lower_bound_ip_address', 'upper_bound_ip_address', 'valid_match'])
print(fraud_df['country'].value_counts().head(10))

valid_match
True     129146
False     21966
Name: count, dtype: int64
country
United States        58049
Unknown              21966
China                12038
Japan                 7306
United Kingdom        4490
Korea Republic of     4162
Germany               3646
France                3161
Canada                2975
Brazil                2961
Name: count, dtype: int64


In [33]:
# Fraud rate by country, for countries with meaningful volume
country_fraud = fraud_df.groupby('country')['class'].agg(['mean', 'count'])
country_fraud = (country_fraud[country_fraud['count'] >= 100]) * 100  # filter out tiny, unreliable samples
country_fraud = country_fraud.sort_values('mean', ascending=False)
print(country_fraud.head(15))

                           mean  count
country                               
Ecuador               26.415094  10600
Tunisia               26.271186  11800
Peru                  26.050420  11900
Ireland               22.916667  24000
New Zealand           22.302158  27800
Saudi Arabia          18.939394  26400
Denmark               15.918367  49000
Chile                 15.347722  41700
Greece                14.285714  23100
United Arab Emirates  14.035088  11400
Belgium               13.691932  40900
Egypt                 13.370474  35900
Venezuela             13.147410  25100
Norway                12.972085  60900
Hong Kong             12.951168  47100


### Findings
- Ecuador (26.4%), Tunisia (26.3%), Peru (26.1%)

I am not claiming "people from Ecuador are fraudulent" — that's a lazy and unfair reading. The more defensible interpretation is that transactions where the IP-derived country doesn't match the customer's actual expected location are often fraud indicators — VPN use, proxy IPs, or account takeover from a different region than the legitimate account holder normally uses. The country itself isn't the cause; it's a proxy for "this transaction's origin looks anomalous relative to where this platform's typical customers are."

In [34]:
print(fraud_df[fraud_df['country'] == 'Unknown']['class'].mean())

0.08572339069471001
